# Tron CSV dataset analysis
Install notebook dependencies with `%pip install "pandas>=2,<4" "networkx>=3,<4" "pyarrow>=14"`.
Set DRIVE_ROOT to the synced dataset folder. Local folders work without mounting Drive.


In [ ]:
from pathlib import Path
import pandas as pd


def colab_load_dataset(drive_root, compressed=False):
    """Mount Colab Drive when needed; load five append logs as string DataFrames.

    Local paths work without Colab. compressed=True prefers gzip, then Parquet
    for each table; False requires raw CSV. No type inference or deduplication
    occurs here: addresses, large integers, leading zeros and blanks are kept.
    """
    if type(compressed) is not bool:
        raise ValueError("compressed must be a boolean")
    root = Path(drive_root).expanduser().resolve()
    mountpoint = Path("/content/drive")
    if root.is_relative_to(mountpoint) and not (mountpoint / "MyDrive").is_dir():
        try:
            from google.colab import drive
        except ImportError as error:
            raise RuntimeError("This Drive path requires Google Colab; use a local dataset path outside Colab.") from error
        drive.mount(str(mountpoint))
    headers = {
        "seed_wallets.csv": [
            "wallet_address",
            "hex_address",
            "transaction_id",
            "source_name",
            "label",
            "created_at"
        ],
        "accounts.csv": [
            "address",
            "hex_address",
            "first_seen_timestamp",
            "last_seen_timestamp",
            "created_date",
            "created_at"
        ],
        "transactions.csv": [
            "tx_hash",
            "block_number",
            "block_timestamp",
            "from_address",
            "to_address",
            "token_symbol",
            "contract_address",
            "amount",
            "fee_trx",
            "created_at"
        ],
        "wallet_hops.csv": [
            "hop_id",
            "wallet_address",
            "seed_address",
            "parent_address",
            "via_tx_hash",
            "hop_level",
            "direction",
            "discovered_at"
        ],
        "pipeline_crawl_state.csv": [
            "address",
            "seed_address",
            "hop_level",
            "synced",
            "expanded",
            "is_supernode",
            "created_at",
            "updated_at"
        ]
    }
    dfs = {}
    for filename, columns in headers.items():
        stem = Path(filename).stem
        names = [filename + ".gz", stem + ".parquet"] if compressed else [filename]
        path = next((root / name for name in names if (root / name).is_file()), None)
        if path is None:
            raise FileNotFoundError(f"Missing dataset table {stem}; expected {' or '.join(names)} in {root}")
        if path.suffix == ".parquet":
            frame = pd.read_parquet(path).fillna("").astype(str)
        else:
            frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        if list(frame.columns) != columns:
            raise ValueError(f"Unexpected schema in {path.name}; expected {columns}")
        dfs[stem] = frame
    return dfs


In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pipeline_data"
dfs = colab_load_dataset(DRIVE_ROOT, compressed=False)
{name: len(frame) for name, frame in dfs.items()}
